# Pipeline V8 — Fase 4: Augmentação por Simetria (152 → 608 NPZs)

**Spec**: `specs/004-melhoria-geracao-dados-cnn/` (T-A3-010).

Gera 3 variantes simétricas de cada NPZ original, sufixadas:
- `dataset_pequeno_NNNN_refH.npz` — reflexão horizontal (c → n_cols−1−c)
- `dataset_pequeno_NNNN_refV.npz` — reflexão vertical (r → n_rows−1−r)
- `dataset_pequeno_NNNN_r180.npz` — rotação 180° (r,c → n_rows−1−r, n_cols−1−c)

**Idempotência garantida**: começa **deletando** todos os arquivos sufixados
antes de regenerar. Re-execuções não criam duplicatas.

**Pré-requisito**: T-A3-008 — módulo `permutacoes_simetria_pontinhos.py`
com função `aplicar_simetria(estado, sym_id) -> estado`.

**Schema**: cada variante contém os mesmos campos do NPZ original, com
`estados`, `canais`, `score_melhor_jogada` e `melhor_jogada` transformados.
Campo `canais[:, :, :, 11]` (paridade) preservado bit-a-bit.

In [ ]:
# Imports
import os
import sys
import glob
import time
import numpy as np

ROOT = os.environ.get('ARENA_SAGAZ_BACKEND', os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..')))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# T-A3-008: módulo de permutações de simetria.
from gerador_dados.jogo_pontinhos.permutacoes_simetria_pontinhos import aplicar_simetria

print(f'Backend root = {ROOT}')
print('Imports OK.')

In [ ]:
# Configuração

DIR_NPZ = os.path.join(ROOT, 'dados', 'profundidade_minimax_11_v7_adaptativo')

# Mapeamento sym_id → sufixo do arquivo gerado.
# sym_id=0: identidade (não gera arquivo)
# sym_id=1: reflexão horizontal
# sym_id=2: reflexão vertical
# sym_id=3: rotação 180°
SIMETRIAS = [
    (1, '_refH'),
    (2, '_refV'),
    (3, '_r180'),
]

print(f'DIR_NPZ  = {DIR_NPZ}')
print(f'Simetrias: {[(s, sf) for s, sf in SIMETRIAS]}')

In [ ]:
# Passo 1: DELETAR todos os arquivos sufixados (idempotência).

sufixos = [sf for _, sf in SIMETRIAS]
a_deletar = [
    f for f in glob.glob(os.path.join(DIR_NPZ, 'dataset_pequeno_*.npz'))
    if any(os.path.splitext(os.path.basename(f))[0].endswith(s) for s in sufixos)
]

print(f'Arquivos sufixados a deletar: {len(a_deletar)}')
for f in a_deletar:
    os.remove(f)
print('Deletados. Pronto para regenerar.')

In [ ]:
# Passo 2: Descobrir NPZs originais.

SUFIXOS_SIMETRIA = tuple(sf for _, sf in SIMETRIAS)
arquivos_originais = sorted([
    f for f in glob.glob(os.path.join(DIR_NPZ, 'dataset_pequeno_*.npz'))
    if not any(os.path.splitext(os.path.basename(f))[0].endswith(s) for s in SUFIXOS_SIMETRIA)
])

print(f'NPZs originais: {len(arquivos_originais)}')

In [ ]:
# Passo 3: Gerar variantes.

def gerar_variante(caminho_orig: str, sym_id: int, sufixo: str) -> dict:
    """Aplica simetria sym_id ao NPZ e salva com sufixo. Sobrescrita atômica."""
    t0 = time.time()
    with np.load(caminho_orig, allow_pickle=False) as f:
        d = dict(f)

    d_sim = aplicar_simetria(d, sym_id)

    base, ext = os.path.splitext(caminho_orig)
    caminho_saida = base + sufixo + ext
    tmp = caminho_saida + '.tmp.npz'
    np.savez_compressed(tmp, **d_sim)
    os.replace(tmp, caminho_saida)

    return {'origem': caminho_orig, 'destino': caminho_saida, 'tempo_s': time.time() - t0}

relatorio = []
t_inicio = time.time()
for arq in arquivos_originais:
    for sym_id, sufixo in SIMETRIAS:
        info = gerar_variante(arq, sym_id, sufixo)
        relatorio.append(info)

print(f'Variantes geradas : {len(relatorio)}')
print(f'Tempo total       : {time.time() - t_inicio:.1f}s')

In [ ]:
# Auditoria final.
# Gate T-A3-010: para cada NPZ original (152), existem 3 sufixados.
# Total de arquivos = 152 × 4 = 608.

todos = sorted(glob.glob(os.path.join(DIR_NPZ, 'dataset_pequeno_*.npz')))
n_total = len(todos)
n_originais = sum(
    1 for f in todos
    if not any(os.path.splitext(os.path.basename(f))[0].endswith(s) for s in SUFIXOS_SIMETRIA)
)
n_sufixados = n_total - n_originais

print(f'Originais : {n_originais}')
print(f'Sufixados : {n_sufixados}  (esperado {n_originais * len(SIMETRIAS)})')
print(f'Total     : {n_total}  (esperado {n_originais * (1 + len(SIMETRIAS))})')

# Verificar que cada original tem os 3 sufixos.
originais = [f for f in todos
             if not any(os.path.splitext(os.path.basename(f))[0].endswith(s)
                        for s in SUFIXOS_SIMETRIA)]
erros = []
for orig in originais:
    base = os.path.splitext(orig)[0]
    for _, sf in SIMETRIAS:
        esperado = base + sf + '.npz'
        if not os.path.exists(esperado):
            erros.append(f'Faltando: {os.path.basename(esperado)}')

if erros:
    print(f'FALHA: {len(erros)} arquivo(s) ausente(s):')
    for e in erros[:10]:
        print(f'  {e}')
else:
    print(f'OK: {n_total} NPZs ({n_originais} originais × 4 = {n_originais * 4} esperados).')
    print('Pronto para Treinamento_CNN_Pontinhos_V8.ipynb.')